# **Apple Product's Pricing Trends Case Study**

This case study analyzes daily pricing dynamics and market trends across Apple's primary hardware product lines (iPhones, iPads, MacBooks, and Apple Watches) from January 2020 to July 2026. The primary dataset comprises 80,000 retail records tracked across Amazon and Flipkart, capturing historical price fluctuations, product specifications, discount metrics, stock availability, ratings, and customer reviews.

The secondary dataset integrates macroeconomic indicators and supply-chain cost drivers, including raw material commodity prices (aluminum and copper futures), foreign exchange rates (USD/EUR, USD/CNY, USD/INR), and global market indices. Together, these datasets enable a comprehensive analysis of e-commerce pricing patterns, product depreciation curves, promotional sale events (e.g., Prime Day, Black Friday, Big Billion Days), and macroeconomic cost pass-through.


## **Dataset Feature Descriptions**

### **1. Retail Pricing Dataset**
1. **Date (datetime):** The daily record observation date ranging from 2020 to 2026.
2. **Platform (categorical):** E-commerce retail channel (Amazon vs. Flipkart).
3. **Product_Category (categorical):** Broad hardware category (iPhone, iPad, MacBook, Watch).
4. **Model_Name (categorical):** Specific Apple hardware model variant.
5. **Condition (categorical):** Unit condition state (New).
6. **Launch_Price_USD (numeric):** Official baseline launch price in US Dollars.
7. **Launch_Price_INR (numeric):** Official baseline launch price converted to Indian Rupees.
8. **Current_Price_USD (numeric):** Observed daily retail price in US Dollars.
9. **Current_Price_INR (numeric):** Observed daily retail price in Indian Rupees.
10. **Discount_Pct (numeric):** Percentage deviation from MSRP/launch price.
11. **Sale_Event (categorical):** Associated major promotional shopping event, if applicable.
12. **Stock_Status (categorical):** Inventory availability status (In Stock / Out of Stock).
13. **Rating (numeric):** Product customer rating score.
14. **Reviews_Count (numeric):** Total volume of user reviews accrued.

### **2. Macro & Cost Drivers Dataset**
1. **Date (datetime):** Monthly aggregated macroeconomic observation date.
2. **Aluminum_Futures (numeric):** Benchmark pricing index for aluminum commodities.
3. **USD_EUR (numeric):** Foreign exchange rate for US Dollar to Euro conversion.
4. **Copper_Futures (numeric):** Benchmark pricing index for copper commodities.
5. **USD_CNY (numeric):** Foreign exchange rate for US Dollar to Chinese Yuan conversion.
6. **USD_INR (numeric):** Foreign exchange rate for US Dollar to Indian Rupee conversion.
7. **Global_Commodity_Index (numeric):** Broad index tracking global raw material cost trends.


## **Key Analytical Objectives**

### **1. Depreciation & Product Lifecycle Dynamics**
Track the velocity of price decay across hardware categories following new generation launches. Measure how historical models adjust in retail pricing once successive hardware iterations enter the market.

### **2. Promotional Elasticity & Event Impact**
Evaluate the magnitude and frequency of discounts during major annual e-commerce sale windows. Identify which product lines receive the deepest markdowns on Amazon vs. Flipkart.

### **3. Macro & Currency Pass-Through Analysis**
Assess the relationship between foreign exchange shifts, raw material inflation indices, and local retail pricing variances to understand pricing resilience against supply chain pressures.

In [1]:
pip install yfinance pandas-datareader wbgapi

Note: you may need to restart the kernel to use updated packages.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import wbgapi as wb
import seaborn as sns
import yfinance as yf
from datetime import datetime
from matplotlib import pyplot as plt
import pandas_datareader.data as web


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

start_date = "2020-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")

print("Fetching Macro & Supply Chain Data...")

# 1. Commodity Prices & FX Rates via yfinance (Daily -> Resampled to Monthly)

# Tickers: 
# ALI=F (Aluminum Futures - iPhone/Mac casing)
# HG=F (Copper Futures - Logic boards/Circuitry)
# CNY=X (USD/CNY - China Manufacturing FX)
# INR=X (USD/INR - India Assembly FX)
# EURUSD=X (USD/EUR - European Retail FX)
yf_tickers = ['ALI=F', 'HG=F', 'CNY=X', 'INR=X', 'EURUSD=X']

yf_raw = yf.download(yf_tickers, start=start_date, end=end_date)['Close']
# Resample daily price feeds to monthly average
yf_monthly = yf_raw.resample('ME').mean()
yf_monthly.columns = ['Aluminum_Futures', 'USD_EUR', 'Copper_Futures', 'USD_CNY', 'USD_INR']

# 2. Economic & Semiconductor Cost Indices via FRED (St. Louis Fed)

# Series:
# PCU33441334413: PPI - Semiconductor & Electronic Component Manufacturing
# PALLFNFINDEXM: Global Price Index of All Commodities
# CHINAMFGWAGE: Manufacturing Wage proxies / indicators where available
fred_series = {
    'PCU33441334413': 'Semiconductor_PPI',
    'PALLFNFINDEXM': 'Global_Commodity_Index'
}

fred_data = {}
for code, name in fred_series.items():
    try:
        df_fred = web.DataReader(code, 'fred', start_date, end_date)
        fred_data[name] = df_fred[code]
    except Exception as e:
        print(f"Could not fetch {code} from FRED: {e}")

fred_df = pd.DataFrame(fred_data).resample('ME').mean()

# 3. Country-Level Macro Trends via World Bank API (wbgapi)

# Indicators:
# NV.IND.MANF.CD: Manufacturing, value added (current US$) for CHN, IND, VNM
# FP.CPI.TOTL.ZG: Inflation, consumer prices (annual %)
wb_indicators = {
    'NV.IND.MANF.CD': 'Mfg_Value_Added_USD',
    'FP.CPI.TOTL.ZG': 'Inflation_Annual_Pct'
}

wb_raw = wb.data.DataFrame(
    list(wb_indicators.keys()), 
    economy=['CHN', 'IND', 'VNM'], 
    time=range(2020, 2026), 
    labels=True
).reset_index()

# # 4. Merge All Macro Signals into One Monthly Panel Dataset

macro_monthly = yf_monthly.join(fred_df, how='outer')
macro_monthly.index.name = 'Date'

# Clean and Save
macro_monthly.to_csv("apple_macro_cost_factors_2020_2026.csv")
print("Data successfully fetched and saved to 'apple_macro_cost_factors_2020_2026.csv'")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ashyou09/apple-products-pricing-dataset-2020-2026/apple_products_pricing_2020_2026.csv
Fetching Macro & Supply Chain Data...


/tmp/ipykernel_16/3884650357.py:38: FutureWarning: YF.download() has changed argument auto_adjust default to True
  yf_raw = yf.download(yf_tickers, start=start_date, end=end_date)['Close']
[*********************100%***********************]  5 of 5 completed


Could not fetch PCU33441334413 from FRED: Unable to read URL: https://fred.stlouisfed.org/graph/fredgraph.csv?id=PCU33441334413
Response Text:
b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n    <meta charset="utf-8">\r\n    <meta http-equiv="X-UA-Compatible" content="IE=edge">\r\n    <meta name="viewport" content="width=device-width, initial-scale=1">\r\n    <title>Error - St. Louis Fed</title>\r\n    <meta name="description" content="">\r\n    <meta name="keywords" content="">    \r\n    <link rel="stylesheet" type="text/css" href="/assets/bootstrap/dist/css/bootstrap.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/css/home.min.css?1553087253">\r\n    <link rel="stylesheet" type="text/css" href="/assets/fontawesome-free/css/all.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/assets/select2/dist/css/select2.min.css">\r\n    <style>p {\r\n        margin-bottom: 1.5em;\r\n    }</style>\r\n</head>\r\n<body>\r\n<link rel="preconnect" href="https://fonts.go

In [3]:
price_data = pd.read_csv('/kaggle/input/datasets/ashyou09/apple-products-pricing-dataset-2020-2026/apple_products_pricing_2020_2026.csv')
macro_data = pd.read_csv('/kaggle/working/apple_macro_cost_factors_2020_2026.csv')

print(f"Price dataset : {price_data.shape}")
print(f"Macro Features dataset : {macro_data.shape}")

Price dataset : (80000, 14)
Macro Features dataset : (79, 7)


In [4]:
price_data.head()

,Date,Platform,Product_Category,Model_Name,Condition,Launch_Price_USD,Launch_Price_INR,Current_Price_USD,Current_Price_INR,Discount_Pct,Sale_Event,Stock_Status,Rating,Reviews_Count
0,2020-09-19,Flipkart,Watch,Apple Watch Series 6 (44mm),New,429,42042,435.81,43322.41,-1.6,NaN,In Stock,4.7,40
1,2020-09-20,Flipkart,Watch,Apple Watch Series 6 (44mm),New,429,42042,436.49,42320.43,-1.7,NaN,Out of Stock,4.6,84
2,2020-09-23,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,422.73,40879.36,1.5,NaN,In Stock,4.4,110
3,2020-09-23,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,425.00,42008.70,0.9,NaN,In Stock,4.8,111
4,2020-09-24,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,436.22,41984.28,-1.7,NaN,In Stock,4.7,35


In [5]:
macro_data.head()

,Date,Aluminum_Futures,USD_EUR,Copper_Futures,USD_CNY,USD_INR,Global_Commodity_Index
0,2020-01-31,1797.238095,6.925543,1.111180,2.749667,71.250036,118.820723
1,2020-02-29,1718.539474,6.995415,1.091091,2.583816,71.548475,110.154758
2,2020-03-31,1628.897727,7.021045,1.107309,2.367136,74.678100,93.175349
3,2020-04-30,1484.916667,7.071005,1.087575,2.306310,76.501459,85.053367
4,2020-05-31,1488.787500,7.099990,1.089994,2.387150,75.848957,91.725813


In [6]:
price_data.isnull().sum()

Date                     0
Platform                 0
Product_Category         0
Model_Name               0
Condition                0
Launch_Price_USD         0
Launch_Price_INR         0
Current_Price_USD        0
Current_Price_INR        0
Discount_Pct             0
Sale_Event           73351
Stock_Status             0
Rating                   0
Reviews_Count            0
dtype: int64

In [7]:
macro_data.isnull().sum()

Date                      0
Aluminum_Futures          0
USD_EUR                   0
Copper_Futures            0
USD_CNY                   0
USD_INR                   0
Global_Commodity_Index    1
dtype: int64

In [8]:
price_data['Sale_Event'] = price_data['Sale_Event'].fillna('Regular Day')

## **Exploratory Data Analysis & Feature Engineering**